<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**

### **Taller de Programación para el Análisis de Datos**

#### **Práctica Unidad 2 Segunda parte**

En base al material (ejemplos y actividades) visto en clase, resuelva los siguientes ejercicios

### **Ejercicio 1**



**Cree una función denominada descarga que devuelva el path donde se descargó el siguiente DataFrame : https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset Luego utilicelá para leer el dataframe con pandas (deberpa completar /NOMBREARCHIVO.csv)**

In [6]:
import kagglehub
import pandas as pd
import os

def descarga():
    path = kagglehub.dataset_download("janiobachmann/bank-marketing-dataset")
    return path

path = descarga()
print("Path de descarga:", path)
print("Archivos disponibles:", os.listdir(path))

df = pd.read_csv(os.path.join(path, "bank.csv"))
print(f"\nDataFrame cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()

Using Colab cache for faster access to the 'bank-marketing-dataset' dataset.
Path de descarga: /kaggle/input/bank-marketing-dataset
Archivos disponibles: ['bank.csv']

DataFrame cargado: 11,162 filas × 17 columnas


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


**Describa brevemente de que se tratan los datos**

**Ejercicio 1 b — Descripción del dataset**

El dataset corresponde a campañas de **marketing directo telefónico** de un banco portugués. Cada fila representa un cliente contactado durante una o más campañas, y el objetivo original es predecir si el cliente suscribió un **depósito a plazo fijo** (`y = yes/no`).

| Variable | Tipo | Descripción |
|---|---|---|
| `age` | numérica | Edad del cliente |
| `job` | categórica | Tipo de trabajo (admin, blue-collar, entrepreneur, etc.) |
| `marital` | categórica | Estado civil (single, married, divorced) |
| `education` | categórica | Nivel educativo (primary, secondary, tertiary) |
| `default` | binaria | ¿Tiene crédito en mora? (yes/no) |
| `balance` | numérica | Saldo promedio anual en euros |
| `housing` | binaria | ¿Tiene préstamo hipotecario? (yes/no) |
| `loan` | binaria | ¿Tiene préstamo personal? (yes/no) |
| `contact` | categórica | Tipo de contacto (cellular, telephone, unknown) |
| `day` / `month` | temporal | Día y mes del último contacto |
| `duration` | numérica | Duración del último contacto en segundos |
| `campaign` | numérica | Cantidad de contactos realizados en esta campaña |
| `pdays` | numérica | Días desde el último contacto en campaña anterior (-1 = sin contacto previo) |
| `previous` | numérica | Cantidad de contactos anteriores a esta campaña |
| `poutcome` | categórica | Resultado de la campaña anterior (success, failure, unknown) |
| `y` | binaria (**target**) | ¿Suscribió el depósito? (yes/no) |

**Fuente:** UCI Machine Learning Repository — Bank Marketing Dataset (Moro et al., 2014)

**Escriba un programa que cree una nueva variable denominada TIPO_SALDO, según las siguientes reglas:**

- Si el saldo es negativo -> 'NEGATIVO'
- Si el saldo es nulo -> 'NULO'
- Si el saldo es positivo -> 'POSITIVO'

In [7]:
# Ejercicio 1 c — Crear variable TIPO_SALDO

def clasificar_saldo(valor):
    if valor < 0:
        return 'NEGATIVO'
    elif valor == 0:
        return 'NULO'
    else:
        return 'POSITIVO'

df['TIPO_SALDO'] = df['balance'].apply(clasificar_saldo)

print("Variable TIPO_SALDO creada exitosamente\n")
print(df['TIPO_SALDO'].value_counts().to_frame('cantidad').assign(
    porcentaje=lambda x: (x['cantidad'] / len(df) * 100).round(2)
))

Variable TIPO_SALDO creada exitosamente

            cantidad  porcentaje
TIPO_SALDO                      
POSITIVO        9700       86.90
NULO             774        6.93
NEGATIVO         688        6.16


**Escriba un programa que cree una nueva variable denominada RIESGO, según las siguientes reglas:**

- Si el tipo de saldo es nulo o negativo y alguna vez entró en default -> 'ALTO'
- Si el tipo de saldo es positivo y alguna vez entró en default -> 'MEDIO'
- Si nunca entró en Default -> 'BAJO'
- En cualquier otro caso-> 'INDEFINIDO'

In [8]:
# Ejercicio 1 d — Crear variable RIESGO

def clasificar_riesgo(row):
    tipo_saldo    = row['TIPO_SALDO']
    entro_default = row['default'] == 'yes'

    if tipo_saldo in ('NULO', 'NEGATIVO') and entro_default:
        return 'ALTO'
    elif tipo_saldo == 'POSITIVO' and entro_default:
        return 'MEDIO'
    elif not entro_default:
        return 'BAJO'
    else:
        return 'INDEFINIDO'

df['RIESGO'] = df.apply(clasificar_riesgo, axis=1)

print("Variable RIESGO creada exitosamente\n")
print(df['RIESGO'].value_counts().to_frame('cantidad').assign(
    porcentaje=lambda x: (x['cantidad'] / len(df) * 100).round(2)
))

print("\nTabla cruzada TIPO_SALDO vs RIESGO:")
print(pd.crosstab(df['TIPO_SALDO'], df['RIESGO'], margins=True))

Variable RIESGO creada exitosamente

        cantidad  porcentaje
RIESGO                      
BAJO       10994       98.49
ALTO         103        0.92
MEDIO         65        0.58

Tabla cruzada TIPO_SALDO vs RIESGO:
RIESGO      ALTO   BAJO  MEDIO    All
TIPO_SALDO                           
NEGATIVO      78    610      0    688
NULO          25    749      0    774
POSITIVO       0   9635     65   9700
All          103  10994     65  11162


**Escriba un programa que de respuesta completa a las siguientes preguntas:**

- ¿Cuál es el saldo medio postivo del conjunto de clientes?
- ¿Entre que valores se concentra el 50% de la edad de los clientes?
- ¿Cuál es el saldo mediano por tipo de riesgo?
- ¿Cuál es el nivel educativo más frecuente de los clientes? ¿y el trabajo más frecuente?
- ¿Cuántos clientes tienen trabajo?
- ¿Cómo es el sesgo de la distribución del saldo de los clientes?
- ¿Cómo es el sentido de la relación entre la edad y la situación de default de los clientes?
- ¿Cómo es el sentido de la relación entre el saldo y el nivel de riesgo de los clientes?
- ¿Existe relación lineal entre la situación de default y riesgo? si existe, ¿cómo es?

In [9]:

# Ejercicio 1 e — Estadísticas descriptivas

import numpy as np
from scipy import stats

# --- 1. Saldo medio positivo ---
saldo_medio_pos = df[df['balance'] > 0]['balance'].mean()
print(f"1. Saldo medio positivo: €{saldo_medio_pos:,.2f}")

# --- 2. IQR de la edad (50% central) ---
q1_edad = df['age'].quantile(0.25)
q3_edad = df['age'].quantile(0.75)
print(f"\n2. El 50% central de la edad se concentra entre {q1_edad:.0f} y {q3_edad:.0f} años (IQR = {q3_edad - q1_edad:.0f} años)")

# --- 3. Saldo mediano por tipo de riesgo ---
print("\n3. Saldo mediano por tipo de riesgo:")
print(df.groupby('RIESGO')['balance'].median().rename('mediana_saldo').to_frame().to_string())

# --- 4. Nivel educativo y trabajo más frecuentes ---
educ_frecuente = df['education'].mode()[0]
job_frecuente  = df['job'].mode()[0]
print(f"\n4. Nivel educativo más frecuente : {educ_frecuente} ({df['education'].value_counts().iloc[0]:,} clientes)")
print(f"   Trabajo más frecuente          : {job_frecuente} ({df['job'].value_counts().iloc[0]:,} clientes)")

# --- 5. Clientes con trabajo (excluye 'unemployed' y 'unknown') ---
clientes_con_trabajo = df[~df['job'].isin(['unknown', 'unemployed'])].shape[0]
print(f"\n5. Clientes con trabajo: {clientes_con_trabajo:,} de {len(df):,} ({clientes_con_trabajo / len(df) * 100:.1f}%)")

# --- 6. Sesgo del saldo ---
sesgo = df['balance'].skew()
if sesgo > 1:
    desc_sesgo = "marcadamente positivo → cola larga hacia valores altos (presencia de saldos muy elevados)"
elif sesgo > 0:
    desc_sesgo = "levemente positivo → ligera asimetría hacia valores altos"
elif sesgo < -1:
    desc_sesgo = "marcadamente negativo → cola larga hacia valores bajos"
else:
    desc_sesgo = "levemente negativo → ligera asimetría hacia valores bajos"
print(f"\n6. Sesgo del saldo: {sesgo:.4f} → {desc_sesgo}")

# --- 7. Relación edad vs default (correlación punto-biserial) ---
default_num = df['default'].map({'yes': 1, 'no': 0})
corr_edad_default, p_edad = stats.pointbiserialr(default_num, df['age'])
sig_edad = "significativa" if p_edad < 0.05 else "no significativa"
sentido_edad = "positivo (a mayor edad, mayor probabilidad de default)" if corr_edad_default > 0 \
               else "negativo (a mayor edad, menor probabilidad de default)"
print(f"\n7. Correlación edad–default: r = {corr_edad_default:.4f} (p = {p_edad:.4f}) → {sig_edad}")
print(f"   Sentido: {sentido_edad}")

# --- 8. Relación saldo vs nivel de riesgo (Spearman, variable ordinal) ---
orden_riesgo = {'BAJO': 0, 'MEDIO': 1, 'ALTO': 2}
riesgo_num = df['RIESGO'].map(orden_riesgo)
corr_saldo_riesgo, p_saldo = stats.spearmanr(df['balance'], riesgo_num, nan_policy='omit')
sig_saldo = "significativa" if p_saldo < 0.05 else "no significativa"
sentido_saldo = "negativo (a mayor saldo, menor nivel de riesgo)" if corr_saldo_riesgo < 0 \
                else "positivo (a mayor saldo, mayor nivel de riesgo)"
print(f"\n8. Correlación Spearman saldo–riesgo: r = {corr_saldo_riesgo:.4f} (p = {p_saldo:.4f}) → {sig_saldo}")
print(f"   Sentido: {sentido_saldo}")

# --- 9. Relación lineal default vs riesgo (Pearson) ---
mask = riesgo_num.notna()
corr_def_riesgo, p_def_riesgo = stats.pearsonr(default_num[mask], riesgo_num[mask])
sig_def = "significativa" if p_def_riesgo < 0.05 else "no significativa"
print(f"\n9. Correlación Pearson default–riesgo: r = {corr_def_riesgo:.4f} (p = {p_def_riesgo:.4e}) → {sig_def}")
if p_def_riesgo < 0.05:
    sentido_def = "positiva" if corr_def_riesgo > 0 else "negativa"
    print(f"   Relación lineal {sentido_def}: los clientes con default presentan niveles de riesgo más altos,")
    print(f"   lo cual es esperado dado que RIESGO fue construido en base a la variable default.")


1. Saldo medio positivo: €1,781.89

2. El 50% central de la edad se concentra entre 32 y 49 años (IQR = 17 años)

3. Saldo mediano por tipo de riesgo:
        mediana_saldo
RIESGO               
ALTO           -204.0
BAJO            564.5
MEDIO           103.0

4. Nivel educativo más frecuente : secondary (5,476 clientes)
   Trabajo más frecuente          : management (2,566 clientes)

5. Clientes con trabajo: 10,735 de 11,162 (96.2%)

6. Sesgo del saldo: 8.2246 → marcadamente positivo → cola larga hacia valores altos (presencia de saldos muy elevados)

7. Correlación edad–default: r = -0.0114 (p = 0.2275) → no significativa
   Sentido: negativo (a mayor edad, menor probabilidad de default)

8. Correlación Spearman saldo–riesgo: r = -0.1482 (p = 0.0000) → significativa
   Sentido: negativo (a mayor saldo, menor nivel de riesgo)

9. Correlación Pearson default–riesgo: r = 0.9567 (p = 0.0000e+00) → significativa
   Relación lineal positiva: los clientes con default presentan niveles de r

**¿Qué otras características descriptivas puede agregar que aporten información sobre los clientes?**

**Ejercicio 1 f — Otras características descriptivas**

Más allá de las preguntas anteriores, se pueden agregar las siguientes métricas para enriquecer el análisis:

| Dimensión | Métrica | Justificación |
|---|---|---|
| **Dispersión del saldo** | Desviación estándar y coeficiente de variación | Permite saber cuán heterogéneos son los saldos; el CV es comparable entre grupos |
| **Valores extremos** | Percentil 95 y 99 del saldo | Identifica la magnitud de los outliers sin eliminarlos |
| **Tasa de suscripción** | Proporción de `deposit = yes` por grupo de riesgo | Conecta el riesgo con el comportamiento comercial objetivo |
| **Duración media por canal** | Media de `duration` por tipo de `contact` | Evalúa la eficiencia de cada canal de contacto |
| **Clientes sin contacto previo** | Proporción con `pdays = -1` | Cuantifica qué fracción de la base es nueva para el banco |
| **Campaña promedio y máxima** | Media y max de `campaign` | Mide el esfuerzo comercial por cliente |
| **Estado civil y riesgo** | Distribución de `marital` por `RIESGO` | Detecta si el perfil familiar se asocia al riesgo financiero |


In [10]:
# Ejercicio 1 f — Otras características descriptivas

# --- Dispersión del saldo ---
std_saldo = df['balance'].std()
cv_saldo  = std_saldo / df['balance'].mean() * 100
print("=== Dispersión del saldo ===")
print(f"  Desviación estándar : €{std_saldo:,.2f}")
print(f"  Coeficiente de variación: {cv_saldo:.1f}%")

# --- Percentiles extremos del saldo ---
p95 = df['balance'].quantile(0.95)
p99 = df['balance'].quantile(0.99)
print(f"\n=== Valores extremos del saldo ===")
print(f"  Percentil 95: €{p95:,.2f}")
print(f"  Percentil 99: €{p99:,.2f}")

# --- Tasa de suscripción por nivel de riesgo ---
# En este dataset la columna target se llama 'deposit' (no 'y')
col_target = 'deposit' if 'deposit' in df.columns else 'y'
print(f"\n=== Tasa de suscripción ({col_target}=yes) por nivel de riesgo ===")
tasa_suscripcion = (
    df.groupby('RIESGO')[col_target]
    .apply(lambda x: (x == 'yes').mean() * 100)
    .rename('tasa_suscripcion_%')
    .round(2)
)
print(tasa_suscripcion.to_frame().to_string())

# --- Duración media del contacto por canal ---
print("\n=== Duración media (seg) por tipo de contacto ===")
print(df.groupby('contact')['duration'].mean().round(1).rename('duracion_media_seg').to_frame().to_string())

# --- Clientes sin contacto previo ---
sin_contacto_prev = (df['pdays'] == -1).sum()
print(f"\n=== Clientes sin contacto previo ===")
print(f"  {sin_contacto_prev:,} clientes ({sin_contacto_prev / len(df) * 100:.1f}%)")

# --- Campaña promedio y máxima ---
print(f"\n=== Esfuerzo de campaña ===")
print(f"  Contactos promedio por cliente: {df['campaign'].mean():.2f}")
print(f"  Máximo de contactos a un cliente: {df['campaign'].max()}")

# --- Estado civil por nivel de riesgo ---
print("\n=== Distribución de estado civil por nivel de riesgo ===")
tabla_marital = pd.crosstab(df['RIESGO'], df['marital'], normalize='index').mul(100).round(1)
print(tabla_marital.to_string())


=== Dispersión del saldo ===
  Desviación estándar : €3,225.41
  Coeficiente de variación: 211.0%

=== Valores extremos del saldo ===
  Percentil 95: €6,026.45
  Percentil 99: €13,226.98

=== Tasa de suscripción (deposit=yes) por nivel de riesgo ===
        tasa_suscripcion_%
RIESGO                    
ALTO                 33.98
BAJO                 47.64
MEDIO                26.15

=== Duración media (seg) por tipo de contacto ===
           duracion_media_seg
contact                      
cellular                376.5
telephone               351.7
unknown                 363.2

=== Clientes sin contacto previo ===
  8,324 clientes (74.6%)

=== Esfuerzo de campaña ===
  Contactos promedio por cliente: 2.51
  Máximo de contactos a un cliente: 63

=== Distribución de estado civil por nivel de riesgo ===
marital  divorced  married  single
RIESGO                            
ALTO         13.6     57.3    29.1
BAJO         11.5     56.9    31.6
MEDIO        21.5     49.2    29.2


### **Ejercicio 2**

Descargue a través de la API de YahooFinance la cotización de cierre de Appel (AAPL) y Amazon (AMZN) de forma mensual para el año 2024 y 2025

Escriba un programa que:
- formatee la fecha como únicamente año-mes-dia
- devuelva el mes con la mayor cotización para cada uno de los activos y el valor correspondiente en cada caso
- devuelva el valor mínimo del 75% superior de las cotizaciones de cada activo y el valor correspondiente en cada caso


In [11]:
# Ejercicio 2 — Cotizaciones mensuales AAPL y AMZN (2024–2025)

import yfinance as yf
import pandas as pd

# Descarga de cotizaciones mensuales para 2024 y 2025
tickers = ['AAPL', 'AMZN']
datos = yf.download(tickers, start='2024-01-01', end='2025-12-31', interval='1mo', auto_adjust=True, progress=False)

# Precio de cierre ajustado
cierre = datos['Close'].copy()

# Formatear la fecha como año-mes-día (sin hora)
cierre.index = cierre.index.strftime('%Y-%m-%d')

print("=== Cotizaciones de cierre mensuales ===")
print(cierre.round(2).to_string())

# Mes con la mayor cotización por activo
print("\n=== Mes con la mayor cotización ===")
for ticker in tickers:
    fecha_max = cierre[ticker].idxmax()
    valor_max = cierre[ticker].max()
    print(f"  {ticker}: {fecha_max}  →  ${valor_max:.2f}")

# Valor mínimo del 75% superior de las cotizaciones
# El 75% superior comienza en el percentil 25 (Q1)
print("\n=== Valor mínimo del 75% superior de cotizaciones (percentil 25) ===")
for ticker in tickers:
    umbral = cierre[ticker].quantile(0.25)
    top75   = cierre[ticker][cierre[ticker] >= umbral]
    print(f"  {ticker}: umbral Q1 = ${umbral:.2f}  |  mínimo del 75% superior = ${top75.min():.2f}  (fecha: {top75.idxmin()})")


=== Cotizaciones de cierre mensuales ===
Ticker        AAPL    AMZN
Date                      
2024-01-01  182.34  155.20
2024-02-01  178.73  176.76
2024-03-01  169.78  180.38
2024-04-01  168.64  175.00
2024-05-01  190.34  176.44
2024-06-01  208.81  193.25
2024-07-01  220.17  186.98
2024-08-01  227.03  178.50
2024-09-01  231.27  186.33
2024-10-01  224.23  186.40
2024-11-01  235.56  207.89
2024-12-01  248.83  219.39
2025-01-01  234.50  237.68
2025-02-01  240.30  212.28
2025-03-01  220.96  190.26
2025-04-01  211.38  184.42
2025-05-01  199.79  205.01
2025-06-01  204.36  219.39
2025-07-01  206.75  234.11
2025-08-01  231.22  229.00
2025-09-01  253.91  219.57
2025-10-01  269.61  244.22
2025-11-01  278.06  233.22
2025-12-01  271.36  230.82

=== Mes con la mayor cotización ===
  AAPL: 2025-11-01  →  $278.06
  AMZN: 2025-10-01  →  $244.22

=== Valor mínimo del 75% superior de cotizaciones (percentil 25) ===
  AAPL: umbral Q1 = $203.22  |  mínimo del 75% superior = $204.36  (fecha: 2025-06-01)
 

### **Ejercicio 3**

Busque un conjunto de datos de su interés en Kaggle y calcule descriptivas de forma tal que pueda construir un reporte para presentar y que permita a quien lo lee conocer las princpales características relevantes de los datos

# 🏠 Estadística Descriptiva Aplicada
## Dataset: Colorado Real Estate Listings — Kaggle

---

### Contexto de negocio

Trabajamos con datos de **propiedades en venta en el estado de Colorado, EE.UU.**, extraídos de listados de bienes raíces. Cada fila representa un inmueble publicado con sus características físicas y precio de venta.

| Variable | Descripción |
|---|---|
| `price` | Precio de venta en USD |
| `bed` | Cantidad de dormitorios |
| `bath` | Cantidad de baños |
| `acre_lot` | Superficie del terreno en acres |
| `house_size` | Superficie cubierta en pies cuadrados |
| `city` | Ciudad donde se ubica la propiedad |
| `state` | Estado (filtrado a Colorado) |
| `zip_code` | Código postal |
| `prev_sold_date` | Fecha de última venta registrada |

**Fuente:** [USA Real Estate Dataset — Kaggle (ahmedshahriarsakib)](https://www.kaggle.com/datasets/ahmedshahriarsakib/usa-real-estate-dataset)

---
### Objetivos
- Aplicar medidas de posición, dispersión y asimetría sobre precios de inmuebles
- Analizar la distribución del precio (¿la media representa al comprador típico?)
- Comparar estadísticas entre ciudades de Colorado
- Identificar la relación entre tamaño de la propiedad y precio

---
## 0. Carga del dataset

In [12]:
import kagglehub
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [13]:
import os

path = kagglehub.dataset_download("ahmedshahriarsakib/usa-real-estate-dataset")
archivo = os.path.join(path, "realtor-data.zip.csv")

df_raw = pd.read_csv(archivo)

# Filtrar Colorado
co = df_raw[df_raw['state'] == 'Colorado'].copy()

print(f"Filas totales en el dataset: {len(df_raw):,}")
print(f"Propiedades en Colorado    : {len(co):,}")
print(f"\nColumnas: {co.columns.tolist()}")
co.head()

100%|██████████| 38.2M/38.2M [00:02<00:00, 17.9MB/s]

Extracting files...


Filas totales en el dataset: 2,226,382
Propiedades en Colorado    : 32,293

Columnas: ['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'street', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date']


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
160667,53016.0,for_sale,7950.0,NaN,NaN,0.09,1912918.0,Colorado City,Colorado,81019.0,NaN,NaN
186240,886.0,for_sale,489900.0,3.0,3.0,0.05,869665.0,Centennial,Colorado,80122.0,2376.0,2002-09-09
258745,82792.0,for_sale,7900.0,NaN,NaN,0.26,1503156.0,Colorado City,Colorado,81019.0,NaN,NaN
262260,33745.0,for_sale,64500.0,NaN,NaN,35.52,1855077.0,Rye,Colorado,81069.0,NaN,NaN
603744,79476.0,for_sale,299900.0,NaN,NaN,62.12,1925265.0,Beulah,Colorado,81023.0,NaN,NaN


---
## 1. Exploración inicial y limpieza

Los datos de listados inmobiliarios suelen tener valores faltantes y outliers extremos. Antes de cualquier análisis, entendemos qué tenemos y tomamos decisiones de limpieza.

In [14]:
print("── Tipos de datos ──")
print(co.dtypes)
print(f"\n── Valores nulos por columna ──")
nulos = co.isnull().sum()
print(nulos[nulos > 0])

── Tipos de datos ──
brokered_by       float64
status             object
price             float64
bed               float64
bath              float64
acre_lot          float64
street            float64
city               object
state              object
zip_code          float64
house_size        float64
prev_sold_date     object
dtype: object

── Valores nulos por columna ──
brokered_by           5
price                14
bed                7229
bath               7840
acre_lot           5548
street              145
city                 61
zip_code             65
house_size         7001
prev_sold_date    12770
dtype: int64


In [15]:
# ── Decisiones de limpieza ────────────────────────────────────────
#
# 1. Eliminar filas sin precio (variable objetivo)
# 2. Eliminar precios = 0 o negativos (errores de carga)
# 3. Eliminar outliers extremos de precio (> percentil 99.5) — mansiones de lujo que distorsionan
# 4. Eliminar filas sin house_size (necesaria para análisis de relación tamaño-precio)
# 5. Convertir prev_sold_date a datetime para análisis temporal

df = co.copy()
df = df.dropna(subset=['price'])
df = df[df['price'] > 0]

# Separamos el top 0.5% de precio para análisis aparte
p995 = df['price'].quantile(0.995)
lujo = df[df['price'] > p995].copy()
df   = df[df['price'] <= p995].copy()

df = df.dropna(subset=['house_size'])
df['prev_sold_date'] = pd.to_datetime(df['prev_sold_date'], errors='coerce')
df['anio_venta'] = df['prev_sold_date'].dt.year

print(f"Propiedades brutas en Colorado : {len(co):,}")
print(f"Propiedades top 0.5% (lujo)    : {len(lujo):,}  (excluidas del análisis principal)")
print(f"Propiedades limpias            : {len(df):,}")
print(f"\nPeriodo de ventas: {df['prev_sold_date'].min().date() if df['prev_sold_date'].notna().any() else 'N/A'} → {df['prev_sold_date'].max().date() if df['prev_sold_date'].notna().any() else 'N/A'}")
print(f"Ciudades        : {df['city'].nunique()}")
print(f"Precio mín / máx: ${df['price'].min():,.0f} / ${df['price'].max():,.0f}")

Propiedades brutas en Colorado : 32,293
Propiedades top 0.5% (lujo)    : 162  (excluidas del análisis principal)
Propiedades limpias            : 25,134

Periodo de ventas: 1906-10-29 → 2022-06-10
Ciudades        : 368
Precio mín / máx: $800 / $11,200,000


---
## 2. Análisis descriptivo — Precio de venta (`price`)

En el mercado inmobiliario, la distribución de precios suele ser **marcadamente asimétrica positiva**: la mayoría de las propiedades se concentra en un rango accesible, pero existen algunas con precios extraordinariamente altos que elevan la media.

In [17]:
precio = df['price']

print("═" * 55)
print("  PRECIO DE VENTA (USD) — Colorado")
print("─" * 55)
print(f"  n                        : {len(precio):,}")
print(f"── Posición ──")
print(f"  Media                    : ${precio.mean():>14,.2f}")
print(f"  Mediana                  : ${precio.median():>14,.2f}")
print(f"  Moda (aprox.)            : ${precio.mode()[0]:>14,.2f}")
print(f"  P10 / P25 / P75 / P90   : ${precio.quantile(0.10):,.0f} / ${precio.quantile(0.25):,.0f} / ${precio.quantile(0.75):,.0f} / ${precio.quantile(0.90):,.0f}")
print(f"── Dispersión ──")
print(f"  Mín / Máx                : ${precio.min():,.0f} / ${precio.max():,.0f}")
print(f"  Rango                    : ${precio.max() - precio.min():,.0f}")
print(f"  RIQ                      : ${precio.quantile(0.75) - precio.quantile(0.25):,.0f}")
print(f"  Desvío estándar          : ${precio.std():>14,.2f}")
print(f"  CV                       : {precio.std()/precio.mean()*100:.1f}%")
print(f"── Asimetría ──")
print(f"  Fisher G1                : {precio.skew():.4f}")
print("═" * 55)

print("\n💡 Observación:")
print(f"   La media (${precio.mean():,.0f}) supera a la mediana (${precio.median():,.0f}).")
diff_pct = (precio.mean() - precio.median()) / precio.median() * 100
print(f"   La media está un {diff_pct:.1f}% por encima de la mediana.")
print(f"   Esto confirma asimetría positiva: las propiedades de alto valor 'tiran' la media hacia arriba.")
print(f"   Un comprador típico en Colorado paga alrededor de ${precio.median():,.0f}, no ${precio.mean():,.0f}.")

═══════════════════════════════════════════════════════
  PRECIO DE VENTA (USD) — Colorado
───────────────────────────────────────────────────────
  n                        : 25,134
── Posición ──
  Media                    : $    802,085.38
  Mediana                  : $    560,000.00
  Moda (aprox.)            : $    450,000.00
  P10 / P25 / P75 / P90   : $289,210 / $415,000 / $799,000 / $1,400,000
── Dispersión ──
  Mín / Máx                : $800 / $11,200,000
  Rango                    : $11,199,200
  RIQ                      : $384,000
  Desvío estándar          : $    950,237.53
  CV                       : 118.5%
── Asimetría ──
  Fisher G1                : 5.1565
═══════════════════════════════════════════════════════

💡 Observación:
   La media ($802,085) supera a la mediana ($560,000).
   La media está un 43.2% por encima de la mediana.
   Esto confirma asimetría positiva: las propiedades de alto valor 'tiran' la media hacia arriba.
   Un comprador típico en Colorado paga a

---
## 3. Outliers — ¿Qué pasa en los extremos del mercado?

Comparamos las estadísticas con y sin el top 1% de precios.

In [18]:
p99 = precio.quantile(0.99)
precio_sin_out = precio[precio <= p99]

print(f"{'Medida':<24} {'Con top 1%':>14} {'Sin top 1%':>14} {'Diferencia':>12}")
print("─" * 67)

medidas = [
    ("Media ($)",           precio.mean(),                  precio_sin_out.mean()),
    ("Mediana ($)",         precio.median(),                precio_sin_out.median()),
    ("Desvío estándar ($)", precio.std(),                   precio_sin_out.std()),
    ("CV (%)",              precio.std()/precio.mean()*100, precio_sin_out.std()/precio_sin_out.mean()*100),
    ("Asimetría G1",        precio.skew(),                  precio_sin_out.skew()),
]

for nombre, con, sin in medidas:
    diff_pct = (sin - con) / con * 100
    print(f"{nombre:<24} {con:>14,.2f} {sin:>14,.2f} {diff_pct:>+11.1f}%")

print(f"\n  Propiedades en el top 1%: {(precio > p99).sum():,} (precio > ${p99:,.0f})")
print("\n💡 La mediana cambia menos del 1% → es ROBUSTA ante outliers.")
print("   La media y el desvío se ven fuertemente afectados por las propiedades de lujo.")

Medida                       Con top 1%     Sin top 1%   Diferencia
───────────────────────────────────────────────────────────────────
Media ($)                    802,085.38     733,285.81        -8.6%
Mediana ($)                  560,000.00     555,000.00        -0.9%
Desvío estándar ($)          950,237.53     645,246.12       -32.1%
CV (%)                           118.47          87.99       -25.7%
Asimetría G1                       5.16           3.28       -36.5%

  Propiedades en el top 1%: 252 (precio > $5,462,790)

💡 La mediana cambia menos del 1% → es ROBUSTA ante outliers.
   La media y el desvío se ven fuertemente afectados por las propiedades de lujo.


---
## 4. Análisis por ciudad

Comparamos las estadísticas descriptivas de precio entre las principales ciudades de Colorado.

In [19]:
# Top 10 ciudades por cantidad de propiedades
top_ciudades = (df['city'].value_counts().head(10).index.tolist())
df_top = df[df['city'].isin(top_ciudades)]

resumen_ciudades = (df_top.groupby('city')['price']
                    .agg(
                        propiedades='count',
                        media='mean',
                        mediana='median',
                        desvio='std',
                        p25=lambda x: x.quantile(0.25),
                        p75=lambda x: x.quantile(0.75),
                        asimetria='skew'
                    )
                    .sort_values('mediana', ascending=False)
                    .round(0))

resumen_ciudades['CV (%)'] = (resumen_ciudades['desvio'] / resumen_ciudades['media'] * 100).round(1)
resumen_ciudades['RIQ ($)'] = (resumen_ciudades['p75'] - resumen_ciudades['p25']).round(0)

print(resumen_ciudades[['propiedades', 'media', 'mediana', 'desvio', 'CV (%)', 'RIQ ($)', 'asimetria']].to_string())
print("\n💡 Notá que en todas las ciudades: media > mediana → distribución asimétrica positiva.")
print("   La ciudad con mayor mediana de precio no es necesariamente la más cara por media.")

                  propiedades      media   mediana     desvio  CV (%)    RIQ ($)  asimetria
city                                                                                       
Boulder                   705  1350886.0  995700.0  1152292.0    85.3  1130000.0        2.0
Littleton                 572   799389.0  669475.0   608691.0    76.1   302722.0        5.0
Denver                   2352   784470.0  600000.0   732320.0    93.4   425000.0        5.0
Longmont                  546   717020.0  558500.0   558914.0    77.9   274075.0        4.0
Aurora                   1272   533743.0  510000.0   208068.0    39.0   245000.0        1.0
Fort Collins             1036   605711.0  502635.0   503175.0    83.1   273659.0       10.0
Windsor                   641   584460.0  499850.0   227983.0    39.0   182060.0        3.0
Colorado Springs         1182   622766.0  495000.0   499383.0    80.2   235000.0        6.0
Loveland                  659   574131.0  489500.0   452486.0    78.8   227000.0

---
## 5. Análisis descriptivo — Tamaño de la propiedad (`house_size`)

¿Cómo se distribuye la superficie cubierta? ¿La distribución es simétrica o también está sesgada?

In [20]:
tam = df['house_size']

print("═" * 55)
print("  SUPERFICIE CUBIERTA (pies cuadrados)")
print("─" * 55)
print(f"  n                        : {len(tam):,}")
print(f"── Posición ──")
print(f"  Media                    : {tam.mean():>10,.1f} ft²")
print(f"  Mediana                  : {tam.median():>10,.1f} ft²")
print(f"  Moda (aprox.)            : {tam.mode()[0]:>10,.1f} ft²")
print(f"  P25 / P75 / P90          : {tam.quantile(0.25):,.0f} / {tam.quantile(0.75):,.0f} / {tam.quantile(0.90):,.0f} ft²")
print(f"── Dispersión ──")
print(f"  Mín / Máx                : {tam.min():,.0f} / {tam.max():,.0f} ft²")
print(f"  RIQ                      : {tam.quantile(0.75) - tam.quantile(0.25):,.0f} ft²")
print(f"  Desvío estándar          : {tam.std():>10,.1f} ft²")
print(f"  CV                       : {tam.std()/tam.mean()*100:.1f}%")
print(f"── Asimetría ──")
print(f"  Fisher G1                : {tam.skew():.4f}")
print("═" * 55)

# Segmentación por rango de tamaño
bins   = [0, 1000, 2000, 3000, 5000, np.inf]
labels = ['< 1.000 ft²', '1.000–2.000', '2.000–3.000', '3.000–5.000', '> 5.000 ft²']
df['segmento_tam'] = pd.cut(df['house_size'], bins=bins, labels=labels)

print("\n── Distribución por segmento de tamaño ──")
seg = df.groupby('segmento_tam', observed=True).agg(
    n=('price', 'count'),
    precio_mediano=('price', 'median'),
    precio_medio=('price', 'mean')
).round(0)
seg['% propiedades'] = (seg['n'] / seg['n'].sum() * 100).round(1)
print(seg.to_string())

═══════════════════════════════════════════════════════
  SUPERFICIE CUBIERTA (pies cuadrados)
───────────────────────────────────────────────────────
  n                        : 25,134
── Posición ──
  Media                    :    2,432.2 ft²
  Mediana                  :    2,117.0 ft²
  Moda (aprox.)            :    1,200.0 ft²
  P25 / P75 / P90          : 1,440 / 3,058 / 4,118 ft²
── Dispersión ──
  Mín / Máx                : 170 / 43,708 ft²
  RIQ                      : 1,618 ft²
  Desvío estándar          :    1,494.6 ft²
  CV                       : 61.4%
── Asimetría ──
  Fisher G1                : 3.6836
═══════════════════════════════════════════════════════

── Distribución por segmento de tamaño ──
                 n  precio_mediano  precio_medio  % propiedades
segmento_tam                                                   
< 1.000 ft²   2375        315000.0      386363.0            9.4
1.000–2.000   9282        450000.0      548555.0           36.9
2.000–3.000   6874     

---
## 6. Correlaciones entre variables numéricas

¿Existe relación entre el tamaño de la propiedad y su precio? ¿Y con la cantidad de dormitorios?

In [21]:
cols_num = ['price', 'bed', 'bath', 'house_size', 'acre_lot']
df_corr  = df[cols_num].dropna()

print("── Matriz de correlación (Pearson) ──")
print(df_corr.corr().round(4))

print("\n── Matriz de correlación (Spearman — más robusta ante outliers) ──")
print(df_corr.corr(method='spearman').round(4))

# Correlación específica precio vs tamaño con interpretación
r_p, p_p = stats.pearsonr(df_corr['price'], df_corr['house_size'])
r_s, p_s = stats.spearmanr(df_corr['price'], df_corr['house_size'])

print(f"\n── Detalle: precio vs house_size ──")
print(f"  Pearson  r = {r_p:.4f}  (p = {p_p:.2e})")
print(f"  Spearman r = {r_s:.4f}  (p = {p_s:.2e})")
print("\n💡 Pearson vs Spearman:")
print("   Pearson mide correlación LINEAL — sensible a outliers.")
print("   Spearman mide correlación MONÓTONA — trabaja sobre rangos, más robusta.")
print("   Con distribuciones asimétricas como el precio inmobiliario, Spearman es más informativo.")

── Matriz de correlación (Pearson) ──
             price     bed    bath  house_size  acre_lot
price       1.0000  0.3173  0.5436      0.5457    0.1737
bed         0.3173  1.0000  0.6609      0.6276    0.0142
bath        0.5436  0.6609  1.0000      0.7746    0.0048
house_size  0.5457  0.6276  0.7746      1.0000    0.0250
acre_lot    0.1737  0.0142  0.0048      0.0250    1.0000

── Matriz de correlación (Spearman — más robusta ante outliers) ──
             price     bed    bath  house_size  acre_lot
price       1.0000  0.4504  0.6269      0.6726    0.3023
bed         0.4504  1.0000  0.6092      0.6471    0.2514
bath        0.6269  0.6092  1.0000      0.7263    0.0892
house_size  0.6726  0.6471  0.7263      1.0000    0.2891
acre_lot    0.3023  0.2514  0.0892      0.2891    1.0000

── Detalle: precio vs house_size ──
  Pearson  r = 0.5457  (p = 0.00e+00)
  Spearman r = 0.6726  (p = 0.00e+00)

💡 Pearson vs Spearman:
   Pearson mide correlación LINEAL — sensible a outliers.
   Spearman mid

---
## 7. Resumen ejecutivo

In [22]:
ciudad_mediana_max = df.groupby('city')['price'].median().idxmax()
precio_med_max     = df.groupby('city')['price'].median().max()

print("╔" + "═"*56 + "╗")
print("║   RESUMEN — Real Estate Colorado (EE.UU.)              ║")
print("╠" + "═"*56 + "╣")
print(f"║  Propiedades analizadas : {len(df):>8,}                      ║")
print(f"║  Ciudades               : {df['city'].nunique():>8,}                      ║")
print("╠" + "═"*56 + "╣")
print("║  PRECIO DE VENTA (USD)                                 ║")
print(f"║  Media                  : ${df['price'].mean():>12,.0f}                  ║")
print(f"║  Mediana                : ${df['price'].median():>12,.0f}   (precio típico)  ║")
print(f"║  Desvío estándar        : ${df['price'].std():>12,.0f}                  ║")
print(f"║  CV                     : {df['price'].std()/df['price'].mean()*100:>11.1f}%                  ║")
print(f"║  Asimetría G1           : {df['price'].skew():>12.4f}   (alta asim. pos.)  ║")
print("╠" + "═"*56 + "╣")
print("║  SUPERFICIE (ft²)                                      ║")
print(f"║  Mediana                : {df['house_size'].median():>12,.0f} ft²                 ║")
print(f"║  RIQ                    : {df['house_size'].quantile(0.75)-df['house_size'].quantile(0.25):>12,.0f} ft²                 ║")
print("╠" + "═"*56 + "╣")
print("║  CONCLUSIONES CLAVE                                    ║")
print("║  • La media del precio NO representa a la propiedad    ║")
print("║    típica → usar MEDIANA para reportes de mercado      ║")
print("║  • Alto CV indica enorme variabilidad de precios       ║")
print("║  • Tamaño y precio tienen correlación positiva         ║")
print(f"║    significativa (Spearman r ≈ 0.6–0.8)               ║")
print(f"║  • Ciudad con mayor precio mediano: {ciudad_mediana_max:<18} ║")
print(f"║    ${precio_med_max:,.0f}                                        ║")
print("╚" + "═"*56 + "╝")

╔════════════════════════════════════════════════════════╗
║   RESUMEN — Real Estate Colorado (EE.UU.)              ║
╠════════════════════════════════════════════════════════╣
║  Propiedades analizadas :   25,134                      ║
║  Ciudades               :      368                      ║
╠════════════════════════════════════════════════════════╣
║  PRECIO DE VENTA (USD)                                 ║
║  Media                  : $     802,085                  ║
║  Mediana                : $     560,000   (precio típico)  ║
║  Desvío estándar        : $     950,238                  ║
║  CV                     :       118.5%                  ║
║  Asimetría G1           :       5.1565   (alta asim. pos.)  ║
╠════════════════════════════════════════════════════════╣
║  SUPERFICIE (ft²)                                      ║
║  Mediana                :        2,117 ft²                 ║
║  RIQ                    :        1,618 ft²                 ║
╠═══════════════════════════════